# Data Cleaning

This notebook audits the raw churn data, applies the cleaning pipeline, and saves cleaned files for EDA, modeling, feature-importance analysis, business targeting logic, and the Streamlit app.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "scripts"))

from clean_data import clean_dataset

DATA_DIR = PROJECT_ROOT / "data"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
CLEANED_TRAIN_PATH = DATA_DIR / "cleaned_train.csv"
CLEANED_TEST_PATH = DATA_DIR / "cleaned_test.csv"

In [2]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (594194, 21)
Test shape: (254655, 20)


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [3]:
INTERNET_SERVICE_COLS = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
]

def audit_dataset(df, name, has_target=True):
    summary = {
        "rows": len(df),
        "columns": df.shape[1],
        "missing_values": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "duplicate_ids": int(df["id"].duplicated().sum()),
        "negative_numeric_values": int(((df[["tenure", "MonthlyCharges", "TotalCharges"]] < 0).sum()).sum()),
        "bad_phone_rows": int(((df["PhoneService"] == "No") & (df["MultipleLines"] != "No phone service")).sum()),
        "bad_internet_rows": int(((df["InternetService"] == "No") & (~df[INTERNET_SERVICE_COLS].eq("No internet service").all(axis=1))).sum()),
    }
    if has_target:
        summary["target_values"] = df["Churn"].value_counts().to_dict()
    return pd.Series(summary, name=name)

pd.concat([
    audit_dataset(train_df, "train", has_target=True),
    audit_dataset(test_df, "test", has_target=False),
], axis=1)

,train,test
rows,594194,254655.0
columns,21,20.0
missing_values,0,0.0
duplicate_rows,0,0.0
duplicate_ids,0,0.0
negative_numeric_values,0,0.0
bad_phone_rows,0,0.0
bad_internet_rows,0,0.0
target_values,"{'No': 460377, 'Yes': 133817}",NaN


## Cleaning decisions

- Keep all business columns for EDA and the Streamlit app.
- Convert the target `Churn` from `Yes/No` to `1/0` in the training data.
- Preserve customer attributes as readable categories for plotting and downstream business logic.
- Save cleaned train and test files separately.

In [4]:
cleaned_train_df = clean_dataset(train_df, is_train=True)
cleaned_test_df = clean_dataset(test_df, is_train=False)

print(cleaned_train_df.shape)
print(cleaned_test_df.shape)

cleaned_train_df.head()

(594194, 21)
(254655, 20)


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,0
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,0
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,0
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,1
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,1


In [5]:
cleaned_train_df.to_csv(CLEANED_TRAIN_PATH, index=False)
cleaned_test_df.to_csv(CLEANED_TEST_PATH, index=False)

print(f"Saved: {CLEANED_TRAIN_PATH}")
print(f"Saved: {CLEANED_TEST_PATH}")

Saved: /Users/bharathkumarvnaik/Downloads/programing/Data_Scientist_projects/playground-series-s6e3/data/cleaned_train.csv
Saved: /Users/bharathkumarvnaik/Downloads/programing/Data_Scientist_projects/playground-series-s6e3/data/cleaned_test.csv


In [6]:
verification = pd.DataFrame({
    "cleaned_train_missing": cleaned_train_df.isna().sum(),
    "cleaned_test_missing": cleaned_test_df.isna().sum(),
}).fillna(0).astype(int)

print("Cleaned train duplicate rows:", cleaned_train_df.duplicated().sum())
print("Cleaned test duplicate rows:", cleaned_test_df.duplicated().sum())
print("Cleaned train target values:", cleaned_train_df["Churn"].value_counts().to_dict())

verification.T

Cleaned train duplicate rows: 0
Cleaned test duplicate rows: 0
Cleaned train target values: {0: 460377, 1: 133817}


,Churn,Contract,Dependents,DeviceProtection,InternetService,MonthlyCharges,MultipleLines,OnlineBackup,OnlineSecurity,PaperlessBilling,...,PaymentMethod,PhoneService,SeniorCitizen,StreamingMovies,StreamingTV,TechSupport,TotalCharges,gender,id,tenure
cleaned_train_missing,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
cleaned_test_missing,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Ready for next steps

The cleaned data is now saved and ready for:

- EDA to understand churn patterns
- model training for churn prediction
- feature-importance analysis to explain churn drivers
- business rules to identify whom to target
- Streamlit app inputs and churn-risk recommendations